<a href="https://colab.research.google.com/github/amina-nasrin/Graph-Neural-Network/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric

In [ ]:
!pip install torch-sparse -f https://data.pyg.org/whl/torch-$(torch --version | cut -d' ' -f2)+$(python -c "import torch; print(torch.cuda.get_device_properties(0).name.split()[0].lower())" | cut -d'+' -f2).html

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

In [ ]:
dataset = Planetoid(root="data/PubMed", name="PubMed", split="public", transform=NormalizeFeatures())

In [ ]:
g = dataset[0]
print(f'Number of nodes: {g.num_nodes}')
print(f'Number of edges: {g.num_edges}')


print(f'Number of features: {dataset.num_node_features}')
print(f'Number of classes: {dataset.num_classes}')
print(f'Number of training nodes: {g.train_mask.sum()}')
print(f'Number of validation nodes: {g.val_mask.sum()}')
print(f'Number of test nodes: {g.test_mask.sum()}')
print(f'Has self loops?: {g.has_self_loops()}')
print(f'Is directed?: {g.is_directed()}')


Number of nodes: 19717
Number of edges: 88648
Number of features: 500
Number of classes: 3
Number of training nodes: 60
Number of validation nodes: 500
Number of test nodes: 1000
Has self loops?: False
Is directed?: False


In [ ]:
from torch_geometric.loader import NeighborLoader
from torch_geometric.loader import NeighborSampler

from torch_geometric.sampler import NeighborSampler
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
g = g.to(device)
train_loader = NeighborLoader(
    g,
    num_neighbors=[25,10],
    batch_size=128,
    num_workers=0,
    shuffle=False
)
test_loader = NeighborLoader(
    g,
    num_neighbors=[25, 10],  # Sample all neighbors for testing
    batch_size=128,
    input_nodes=None,  # No specific input nodes for testing
    shuffle=False,
    num_workers=0,
)
#test_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=None, sizes=[-1], batch_size=128, shuffle=False, num_workers=0)
#train_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=g.train_mask, sizes=[25, 10], batch_size=128, shuffle=True, num_workers=0)
#test_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=None, sizes=[-1], batch_size=128, shuffle=False, num_workers=0)

In [ ]:
from torch_geometric.nn import SAGEConv

class Minibatch_GraphSAGE(nn.Module):
  def __init__(self, in_dimension, hidden_dimension, num_classes):
    super(Minibatch_GraphSAGE, self).__init__()
    ## 2-layer GraphSAGE model
    self.num_layers = 2
    self.layers = torch.nn.ModuleList()
    ##TODO
    ##=== Append two SAGEConv layers with 'max' aggregator.
    self.layers.append(SAGEConv(in_dimension, hidden_dimension,aggr='max'))
    self.layers.append(SAGEConv(hidden_dimension, num_classes, aggr='max'))
    ##===

  def forward(self, h, adjs):
    for i, (edge_index, _, size) in enumerate(adjs):
      h_target = h[:size[1]]
      h = self.layers[i]((h, h_target), edge_index)
      if i != self.num_layers - 1:
        h = F.relu(h)
    return h.log_softmax(dim=-1)

  def inference(self, h_all):
      for i in range(self.num_layers):
        hs = []
        for _, n_id, adj in test_loader:
          edge_index, _, size = adj.to(device)
          h = h_all[n_id].to(device)
          h_target = h[:size[1]]
          h = self.layers[i]((h, h_target), edge_index)
          if i != self.num_layers - 1:
            h = F.relu(h)
          hs.append(h.log_softmax(dim=-1))

        h_all = torch.cat(hs, dim=0)
      return h_all

def Minibatch_train(g, model):
  optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
  best_val_acc = 0
  best_test_acc = 0
  model.train()
  for epoch in range(1, 101):
    for batch_size, n_id, adjs in train_loader:
      adjs = [adj.to(device) for adj in adjs]
## Forward
      out = model(g.x[n_id], adjs)
## Compute loss
      loss = F.cross_entropy(out, g.y[n_id[:batch_size]])
## Backward
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
## Compute prediction
  model.eval()
  preds = model.inference(g.x).argmax(dim=1)
## Compute accuracy on training/validation/test
  train_acc = (preds[g.train_mask] == g.y[g.train_mask]).float().mean()
  val_acc = (preds[g.val_mask] == g.y[g.val_mask]).float().mean()
  test_acc = (preds[g.test_mask] == g.y[g.test_mask]).float().mean()
## Save the best validation accuracy and the corresponding test accuracy.
  if best_val_acc < val_acc:
    best_val_acc = val_acc
    best_test_acc = test_acc
  if epoch % 10 == 0:
    print('In epoch {}, loss: {:.3f}, val acc: {:.3f} (best {:.3f}), test acc: {:.3f} (best {:.3f})'.format(epoch, loss, val_acc, best_val_acc, test_acc, best_test_acc))
## Create the model with given dimensions
model = Minibatch_GraphSAGE(dataset.num_node_features, 16,
dataset.num_classes).to(device)
## Train the model
Minibatch_train(g, model)


ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:
# Install PyTorch (replace cu117 with your CUDA version or cpu if not using CUDA)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu117

# Install torch-scatter, torch-sparse, torch-cluster, and torch-spline-conv
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.0.0+cu117.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu117.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.0.0+cu117.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.0+cu117.html

# Install torch-geometric
!pip install torch-geometric


Looking in indexes: https://download.pytorch.org/whl/cu117
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 50.8 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 7.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 2.4 MB/s eta 0:00:00


In [ ]:
# Install torch-sparse (replace cu117 with your CUDA version or cpu if needed)
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu117.html


Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html


In [ ]:
!pip install torch-geometric


In [ ]:
!pip show torch-sparse pyg-lib torch-geometric


Name: torch_sparse
Version: 0.6.18
Summary: PyTorch Extension Library of Optimized Autograd Sparse Matrix Operations
Home-page: https://github.com/rusty1s/pytorch_sparse
Author: Matthias Fey
Author-email: matthias.fey@tu-dortmund.de
License: 
Location: /usr/local/lib/python3.10/dist-packages
Requires: scipy
Required-by: 
---
Name: torch-geometric
Version: 2.6.1
Summary: Graph Neural Network Library for PyTorch
Home-page: https://pyg.org
Author: 
Author-email: Matthias Fey <matthias@pyg.org>
License: 
Location: /usr/local/lib/python3.10/dist-packages
Requires: aiohttp, fsspec, jinja2, numpy, psutil, pyparsing, requests, tqdm
Required-by: 


In [ ]:
!pip uninstall torch-geometric

# Reinstall torch-geometric with its full dependencies
!pip install torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu117.html


Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu117.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.8 MB/s eta 0:00:00


In [ ]:
!pip show torch-sparse


Name: torch-sparse
Version: 0.6.18+pt20cu117
Summary: PyTorch Extension Library of Optimized Autograd Sparse Matrix Operations
Home-page: https://github.com/rusty1s/pytorch_sparse
Author: Matthias Fey
Author-email: matthias.fey@tu-dortmund.de
License: 
Location: /usr/local/lib/python3.10/dist-packages
Requires: scipy
Required-by: 


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_sparse import SparseTensor
import torch_sparse
#SparseTensor' requires 'torch-sparse'

dataset = Planetoid(root="data/PubMed", name="PubMed", split="public", transform=NormalizeFeatures())

g = dataset[0]
print(f'Number of nodes: {g.num_nodes}')
print(f'Number of edges: {g.num_edges}')


print(f'Number of features: {dataset.num_node_features}')
print(f'Number of classes: {dataset.num_classes}')
print(f'Number of training nodes: {g.train_mask.sum()}')
print(f'Number of validation nodes: {g.val_mask.sum()}')
print(f'Number of test nodes: {g.test_mask.sum()}')
print(f'Has self loops?: {g.has_self_loops()}')
print(f'Is directed?: {g.is_directed()}')

from torch_geometric.loader import NeighborLoader
#from torch_geometric.loader import NeighborSampler

#from torch_geometric.sampler import NeighborSampler
import torch
import torch_geometric
import torch_sparse

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
g = g.to(device)
train_loader = NeighborLoader(
    g,
    num_neighbors=[25,10],
    batch_size=128,
    num_workers=0,
    shuffle=False
)
'''test_loader = NeighborLoader(
    g,
    num_neighbors=[-1],  # Sample all neighbors for testing
    batch_size=128,
    input_nodes=None,  # No specific input nodes for testing
    shuffle=False,
    num_workers=0,
)'''
#test_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=None, sizes=[-1], batch_size=128, shuffle=False, num_workers=0)
#train_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=g.train_mask, sizes=[25, 10], batch_size=128, shuffle=True, num_workers=0)
#test_loader = torch_geometric.loader.NeighborSampler(g.edge_index, node_idx=None, sizes=[-1], batch_size=128, shuffle=False, num_workers=0)

from torch_geometric.nn import SAGEConv

class Minibatch_GraphSAGE(nn.Module):
  def __init__(self, in_dimension, hidden_dimension, num_classes):
    super(Minibatch_GraphSAGE, self).__init__()
    ## 2-layer GraphSAGE model
    self.num_layers = 2
    self.layers = torch.nn.ModuleList()
    ##TODO
    ##=== Append two SAGEConv layers with 'max' aggregator.
    self.layers.append(SAGEConv(in_dimension, hidden_dimension,aggr='max'))
    self.layers.append(SAGEConv(hidden_dimension, num_classes, aggr='max'))
    ##===

  def forward(self, h, adjs):
    for i, (edge_index, _, size) in enumerate(adjs):
      h_target = h[:size[1]]
      h = self.layers[i]((h, h_target), edge_index)
      if i != self.num_layers - 1:
        h = F.relu(h)
    return h.log_softmax(dim=-1)

  def inference(self, h_all):
      for i in range(self.num_layers):
        hs = []
        for _, n_id, adj in test_loader:
          edge_index, _, size = adj.to(device)
          h = h_all[n_id].to(device)
          h_target = h[:size[1]]
          h = self.layers[i]((h, h_target), edge_index)
          if i != self.num_layers - 1:
            h = F.relu(h)
          hs.append(h.log_softmax(dim=-1))

        h_all = torch.cat(hs, dim=0)
      return h_all

def Minibatch_train(g, model):
  optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
  best_val_acc = 0
  best_test_acc = 0
  model.train()

  for epoch in range(1, 101):

      for batch_size, n_id, adjs in train_loader:
        adjs = [adj.to(device) for adj in adjs]
## Forward
        out = model(g.x[n_id], adjs)
## Compute loss
        loss = F.cross_entropy(out, g.y[n_id[:batch_size]])
## Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
## Compute prediction
  model.eval()
  preds = model.inference(g.x).argmax(dim=1)
## Compute accuracy on training/validation/test
  train_acc = (preds[g.train_mask] == g.y[g.train_mask]).float().mean()
  val_acc = (preds[g.val_mask] == g.y[g.val_mask]).float().mean()
  test_acc = (preds[g.test_mask] == g.y[g.test_mask]).float().mean()
## Save the best validation accuracy and the corresponding test accuracy.
  if best_val_acc < val_acc:
    best_val_acc = val_acc
    best_test_acc = test_acc
  if epoch % 10 == 0:
    print('In epoch {}, loss: {:.3f}, val acc: {:.3f} (best {:.3f}), test acc: {:.3f} (best {:.3f})'.format(epoch, loss, val_acc, best_val_acc, test_acc, best_test_acc))
## Create the model with given dimensions
model = Minibatch_GraphSAGE(dataset.num_node_features, 16,
dataset.num_classes).to(device)
## Train the model
Minibatch_train(g, model)


OSError: /usr/local/lib/python3.10/dist-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev